### Robustness Analysis

### Main Result and Declaration

Declaration- This analysis is descriptive. We estimate the conditional correlation between average monthly petrol prices and public transport patronage in Melbourne between 2022 and 2025, controlling for rainfall, population, public holidays, and seasonal patterns. We do not claim causality as petrol prices are likely correlated with other macroeconomic factors (such as post COVID recovery) that cannot be fully disentag=ngled with the available data.

Main Result-  Using our preferred log-linear OLS specification, we find that a $1 per litre increase in average monthly petrol price is associated with approximately a 64.8% increase in monthly public transport patronage (β₁ = 0.648, HC3-robust SE = 0.324, p < 0.05), holding rainfall, population, public holiday count, and seasonal month fixed effects constant.

Key threat- The most significant threat is OVB from the post-COVID recovery. In early 2022, petrol prices spiked due to global supply shocks and at the same time Melbourne commuters were returning to work as COVID restrictions lifted. Both forces increased patronage simultaneously, meaning the coefficient is likely upward biased. The checks below stress-test whether the result is driven by this period.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import OLSInfluence
import warnings
warnings.filterwarnings('ignore')

# Load clean data
df = pd.read_csv('data/clean/merged_final.csv', parse_dates=['month'])

# Filter to 48 months: January 2022 to December 2025
max_sample_date = pd.Timestamp('2025-12-01')
df = df[df['month'] <= max_sample_date]

# Create total patronage from individual modes
patronage_cols = [
    'Metropolitan train', 'Metropolitan tram', 'Metropolitan bus',
    'Regional train', 'Regional coach', 'Regional bus'
]
df['total_patronage'] = df[patronage_cols].sum(axis=1)

# Log-transform patronage
if 'log_patronage' not in df.columns:
    df['log_patronage'] = np.log(df['total_patronage'])

# Month fixed effects (January = omitted baseline)
df['month_num'] = df['month'].dt.month
month_dummies = pd.get_dummies(df['month_num'], prefix='month', drop_first=True)
df = pd.concat([df, month_dummies], axis=1)

month_fe_cols = [c for c in df.columns if c.startswith('month_')]

print(f"Full sample: {len(df)} months ({df['month'].min().strftime('%b %Y')} to {df['month'].max().strftime('%b %Y')})")

Full sample: 48 months (Jan 2022 to Dec 2025)


### Main Specification

Log-linear OLS with full controls and seasonal fixed effects. HC3 robust standard errors throughout

In [2]:
y_log    = df['log_patronage']
y_levels = df['total_patronage']

base_controls = ['PetrolPrice', 'rainfall_mm', 'population', 'public_holiday_count']
month_fe_float = df[month_fe_cols].astype(float)
X_main = sm.add_constant(df[base_controls].join(month_fe_float))

main_model = sm.OLS(y_log, X_main).fit(cov_type='HC3')

print(f"β(PetrolPrice) = {main_model.params['PetrolPrice']:.6f}")
print(f"SE             = {main_model.bse['PetrolPrice']:.6f}")
print(f"p-value        = {main_model.pvalues['PetrolPrice']:.4f}")
print(f"N = {int(main_model.nobs)}, R² = {main_model.rsquared:.3f}")

β(PetrolPrice) = 0.648278
SE             = 0.324025
p-value        = 0.0454
N = 48, R² = 0.791


### Standard Error Choice

HC3 heteroskedasticity-robust standard errors are used throughout all specifications. With monthly time series data, residual variance is unlikely to be constant- petrol prices and patronage levels both vary considerably across the sample period, this therefore makes heteroskedasticity the expected case here. HC3 is preferred over HC1 or HC2 because of its superior small-sample behaviour- with only N=48 observations, the leverage correction in HC3 is particularly important as individual months can exert disproportionate influence on the estimates. Clustering is not appropriate here as there is only one geographic unit (Melbourne) and no meaningful cluster structure in the data.


### Check 1- Alternative Control Sets

Assumption tested: OVB. Does the petrol price coefficient depend on which controls are included?

Method:We re-estimate the log-linear model three times, progressively adding controls- first petrol price only, then adding rainfall, population, and public holidays, then adding seasonal fixed effects- to test whether the main coefficient is sensitive to the choice of control set.

(2) Minimal: Petrol price only
(3) Controls, no seasonal FE: Petrol price + rainfall + population + public holidays
(1) Main: Full controls + 11 month dummies

In [3]:
# Minimal
X_minimal = sm.add_constant(df[['PetrolPrice']])
model_minimal = sm.OLS(y_log, X_minimal).fit(cov_type='HC3')

# Controls, no seasonal FE
X_controls = sm.add_constant(df[base_controls])
model_controls = sm.OLS(y_log, X_controls).fit(cov_type='HC3')

for label, m in [('(2) Minimal', model_minimal),
                 ('(3) Controls no FE', model_controls),
                 ('(1) Main', main_model)]:
    print(f"{label}: β={m.params['PetrolPrice']:.4f}, "
          f"SE={m.bse['PetrolPrice']:.4f}, "
          f"p={m.pvalues['PetrolPrice']:.3f}, "
          f"N={int(m.nobs)}, R²={m.rsquared:.3f}")

(2) Minimal: β=0.7739, SE=0.5733, p=0.177, N=48, R²=0.186
(3) Controls no FE: β=0.6480, SE=0.4217, p=0.124, N=48, R²=0.545
(1) Main: β=0.6483, SE=0.3240, p=0.045, N=48, R²=0.791


### Check 2- Drop 2022 (Most Impportant Check)

Assumption tested: Whether the result is driven by the confounded 2022 COVID recovery period

Method: We restrict the sample to January 2023 to December 2025 (N=36), dropping the entire 2022 calendar year, to test whether the main result is driven by the confounded COVID recovery period where petrol prices and patronage both spiked simultaneously.

In [4]:
df_no2022 = df[df['month'].dt.year >= 2023].copy()
month_fe_no2022 = [c for c in df_no2022.columns if c.startswith('month_')]

month_fe_no2022_float = df_no2022[month_fe_no2022].astype(float)
X_no2022  = sm.add_constant(df_no2022[base_controls].join(month_fe_no2022_float))
y_no2022  = df_no2022['log_patronage']
model_no2022 = sm.OLS(y_no2022, X_no2022).fit(cov_type='HC3')

print(f"Sample: 2023–2025, N = {int(model_no2022.nobs)}")
print(f"β(PetrolPrice) = {model_no2022.params['PetrolPrice']:.6f}")
print(f"SE             = {model_no2022.bse['PetrolPrice']:.6f}")
print(f"p-value        = {model_no2022.pvalues['PetrolPrice']:.4f}")
print(f"R²             = {model_no2022.rsquared:.3f}")

Sample: 2023–2025, N = 36
β(PetrolPrice) = 0.240791
SE             = 0.141089
p-value        = 0.0879
R²             = 0.892


### Check 3- Levels Specification

Assumption tested: Whether the result depends on the log transformation of the dependent variable

Method: We re-estimate the main specification with total patronage in persons as the dependent variable rather than log(patronage), holding all controls and seasonal fixed effects constant, to test whether the result depends on the log transformation.

In [5]:
model_levels = sm.OLS(y_levels, X_main).fit(cov_type='HC3')

print(f"β(PetrolPrice) = {model_levels.params['PetrolPrice']:,.0f} trips per $1/L")
print(f"SE             = {model_levels.bse['PetrolPrice']:,.0f}")
print(f"p-value        = {model_levels.pvalues['PetrolPrice']:.4f}")
print(f"N = {int(model_levels.nobs)}, R² = {model_levels.rsquared:.3f}")

β(PetrolPrice) = 17,667,943 trips per $1/L
SE             = 7,448,140
p-value        = 0.0177
N = 48, R² = 0.829


### Check 4- Influential Observations (Cook's Distance)

Assumption tested: Whether the result is driven by a small number of high-influence months.

Method: We compute Cook's distance for each observation using the rule of thumb threshold of 4/N = 0.083, flag months that exceed this threshold, and re-estimate the main specification after dropping them. 4 months were flagged as described in the interpretation.

In [6]:
# Check 4 — Influential Observations (Cook's Distance)
influence  = OLSInfluence(main_model)
cooks_d    = influence.cooks_distance[0]
threshold  = 4 / len(df)
high_infl  = cooks_d > threshold

print(f"Threshold (4/N): {threshold:.4f}")
print(f"Months flagged:  {high_infl.sum()}")
print(f"Flagged:         {df.loc[high_infl, 'month'].dt.strftime('%b %Y').tolist()}")

df_clean  = df.loc[~high_infl].copy()

# Fix: convert month dummy columns to int
for col in month_fe_cols:
    df_clean[col] = df_clean[col].astype(int)

X_clean   = sm.add_constant(df_clean[base_controls + month_fe_cols])
y_clean   = df_clean['log_patronage']
model_clean = sm.OLS(y_clean, X_clean).fit(cov_type='HC3')

print(f"\nAfter dropping influential months (N = {int(model_clean.nobs)}):")
print(f"β(PetrolPrice) = {model_clean.params['PetrolPrice']:.6f}")
print(f"SE             = {model_clean.bse['PetrolPrice']:.6f}")
print(f"p-value        = {model_clean.pvalues['PetrolPrice']:.4f}")
print(f"R²             = {model_clean.rsquared:.3f}")

Threshold (4/N): 0.0833
Months flagged:  4
Flagged:         ['Jan 2022', 'Feb 2022', 'Sep 2022', 'Jan 2023']

After dropping influential months (N = 44):
β(PetrolPrice) = 0.243373
SE             = 0.152035
p-value        = 0.1094
R²             = 0.788


In [8]:
def stars(p):
    if p < 0.01: return '***'
    elif p < 0.05: return '**'
    elif p < 0.10: return '*'
    return ''

def fmt(coef, se, pval, dec=4):
    return f"{coef:.{dec}f}{stars(pval)} ({se:.{dec}f})"

def fmt_levels(coef, se, pval):
    return f"{coef:,.0f}{stars(pval)} ({se:,.0f})"

specs = [
    ('(1) Main',              main_model,     'log'),
    ('(2) Minimal',           model_minimal,  'log'),
    ('(3) Controls, no FE',   model_controls, 'log'),
    ('(4) Drop 2022',         model_no2022,   'log'),
    ('(5) Levels',            model_levels,   'levels'),
    ('(6) Drop influential',  model_clean,    'log'),
]

rows = {}

# Petrol price
rows['Petrol Price ($/L)'] = []
for label, m, form in specs:
    c, se, p = m.params['PetrolPrice'], m.bse['PetrolPrice'], m.pvalues['PetrolPrice']
    rows['Petrol Price ($/L)'].append(fmt_levels(c, se, p) if form=='levels' else fmt(c, se, p))

# Controls
for var, lbl in [('rainfall_mm','Rainfall (mm)'),
                 ('population','Population'),
                 ('public_holiday_count','Public Holidays')]:
    rows[lbl] = []
    for label, m, form in specs:
        if var in m.params:
            c, se, p = m.params[var], m.bse[var], m.pvalues[var]
            rows[lbl].append(fmt_levels(c,se,p) if form=='levels' else fmt(c,se,p))
        else:
            rows[lbl].append('—')

# Seasonal FE
rows['Seasonal FE'] = ['Yes' if any(c.startswith('month_') for c in m.params.index)
                        else 'No' for _, m, _ in specs]

# N and R2
rows['N']  = [str(int(m.nobs))       for _, m, _ in specs]
rows['R²'] = [f"{m.rsquared:.3f}"    for _, m, _ in specs]

table = pd.DataFrame(rows, index=[l for l,_,_ in specs]).T
print(table.to_string())
print()
print("Notes: Dependent variable is log(total_patronage) except col. (5) where it is patronage in persons.")
print("HC3 robust SEs in parentheses. * p<0.10, ** p<0.05, *** p<0.01.")
print("(2) petrol price only, no controls or FE. (3) controls, no seasonal FE.")
print("(4) sample restricted to Jan 2023–Dec 2025. (5) levels functional form.")
print("(6) drops months with Cook's distance > 4/N.")

import os; os.makedirs('output', exist_ok=True)
table.to_csv('output/robustness_table.csv')
print("\nSaved to output/robustness_table.csv")

# Summary Interpretation
print("\n=== ROBUSTNESS SUMMARY ===")
print("Main result (β = 0.648**): $1/L petrol increase → 64.8% higher patronage")
print("✓ Survives: Alternative control sets (coefficient stable 0.648-0.774)")
print("✓ Survives: Functional form (log vs levels, though levels coefficient huge)")
print("✗ Fails: 2022 subsample restriction (coefficient drops to 0.241*, loses significance)")
print("✗ Fails: Dropping influential observations (coefficient 0.243, not significant)")
print("\nInterpretation: Result robust to controls/functional form but fragile to 2022 exclusion.")
print("This supports OVB concern from post-COVID recovery confounding petrol prices and patronage.")
print("Credibility: Moderate - correlation exists but likely upward biased by 2022 events.")

                              (1) Main      (2) Minimal (3) Controls, no FE       (4) Drop 2022                (5) Levels (6) Drop influential
Petrol Price ($/L)   0.6483** (0.3240)  0.7739 (0.5733)     0.6480 (0.4217)    0.2408* (0.1411)  17,667,943** (7,448,140)      0.2434 (0.1520)
Rainfall (mm)          0.0001 (0.0008)                —    -0.0001 (0.0005)    -0.0005 (0.0004)           -1,964 (23,060)     -0.0003 (0.0006)
Population          0.0000*** (0.0000)                —  0.0000*** (0.0000)  0.0000*** (0.0000)                 27*** (5)   0.0000*** (0.0000)
Public Holidays       -0.0048 (0.0330)                —    -0.0150 (0.0140)  -0.0289** (0.0116)        -217,499 (849,933)     -0.0181 (0.0267)
Seasonal FE                        Yes               No                  No                 Yes                       Yes                  Yes
N                                   48               48                  48                  36                        48                   44

### Interpretation

Check 1- Alternative Control Sets (Columns 1-3)
The petrol price coefficient moves predictably as controls are added- from β = 0.774 in the minimal specification down to β = 0.648 in the preferred log specification. This pattern is consistent with the OVB formula as omitted population growth and seasonal patterns are positively correlated with both petrol prices and patronage, so stripping them out inflates the coefficient and the effect is overstated. The direction and magnitude of the movement is exactly what our threats discussion predicted. The coefficient stabilises between columns 3 and 1 once seasonal fixed effects are added, and the sign stays the same throughout (positive). The minimal specification is not statistically significant (p = 0.177), which reflects the noise absorbed by the omitted controls rather than an absence of association. Overall this check supports the preferred log specification- this is an indication that the controls are doing the work they were designed to do.

Check 2- Drop 2022 (Column 4)
This is the most important check given the OVB threat. Restricting the sample to 2023-2025 (N = 36) substantially reduces the coefficient from β = 0.648 to β = 0.241, and significance falls to the 10% level (p = 0.088). This is a meaningful change and as discussed in lectures should be reported honestly. It indicates that a portion of the main result is driven by the confounded 2022 period, where the COVID recovery and the global petrol price spike coincided. However, the coefficient does not collapse to zero and retains its positive sign and marginal significance- suggesting the association between petrol prices and patronage is not entirely a product of the 2022 period, but is weaker once that period is removed. This is consistent with our declaration that the result is descriptive and upward biased, and reinforces the caution against causal interpretation.

Check 3- Levels Specification (Column 5)
The levels model produces a positive and statistically significant coefficient (β = 17,667,943, p = 0.018), indicating that a $1 per litre increase in petrol price is associated with approximately 17.7 million additional trips per month. The sign and significance are consistent with the preferred log-linear result, supporting the robustness of the functional form choice. The R² values are not directly comparable across the two models as they explain variation in different transformations of the outcome. The main result holds under this check.

Check 4- Influential Observations
Four months were flagged as high influence- January 2022, February 2022, September 2022, and January 2023- all plausibly related to the COVID recovery period and seasonal transitions. After dropping these months (N = 44), the coefficient falls to β = 0.243 (p = 0.109), losing statistical significance at conventional levels. This mirrors the finding from Check 2 and reinforces the same conclusion: the early 2022 months are doing disproportionate work in the main specification. The result is sensitive to this influential period, which is consistent with our OVB discussion. This is an honest finding that the 64.8% estimate from the main specification should be interpreted as an upper bound on the true conditional correlation, with the 2023-2025 restricted estimate of around 24% likely providing a more reliable signal.

Overall
Across four checks probing distinct assumptions, the main finding is partially robust. The positive sign is preserved in every specification. Statistical significance at the 5% level is specific to the full sample with seasonal controls- once the 2022 period is removed or influential months are dropped, the coefficient falls from 0.648 to around 0.241 and loses significance at conventional levels. This is because the point estimate drops substantially while the smaller sample (N=36) provides less statistical power to detect the association, so the t-statistic weakens. This pattern is informative rather than damaging: it is entirely consistent with the descriptive declaration and the upward bias argument made in the threats section. The surviving association across all checks supports reporting a positive conditional correlation, while the sensitivity to 2022 reinforces that the magnitude should not be interpreted causally.